In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
df = pd.read_csv("insurance.csv")
len(df)

In [ ]:
print("Aperçu brut du CSV :")
display(df.head())          # premières lignes
print("\nInfo :")
print(df.info())            # types & non‑null
print("\nStatistiques descriptives :")
display(df.describe(include="all").T)

In [ ]:
# -------- 3) Encodage one‑hot des variables catégorielles -------
df = pd.get_dummies(
    df,
    columns=["sex", "smoker", "region"],
    drop_first=True        # évite la redondance linéaire
)
dataset = df
dataset.head()

In [ ]:
# -------- 4) Séparation train / test (80 / 20) ------------------
train_ds, test_ds = train_test_split(df, test_size=0.2, random_state=42)

train_labels = train_ds.pop("expenses")   # on retire la cible
test_labels  = test_ds.pop("expenses")

In [ ]:
# -------- 5) Normalisation des colonnes numériques --------------
numeriques = ["age", "bmi", "children"]
moy, ec = train_ds[numeriques].mean(), train_ds[numeriques].std()

train_ds[numeriques] = (train_ds[numeriques] - moy) / ec
test_ds[numeriques]  = (test_ds[numeriques]  - moy) / ec

train_ds = train_ds.astype("float32")
test_ds  = test_ds.astype("float32")

In [ ]:
# -------- 6) Modèle Keras ---------------------------------------
normalizer = layers.Normalization()
normalizer.adapt(np.array(train_ds))

model = keras.Sequential([
    normalizer,
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(1)                       # sortie scalaire
])

model.compile(
    optimizer=keras.optimizers.Adam(0.01),
    loss="mse",
    metrics=["mae", "mse"]
)

print("\nRésumé du modèle :")
model.build(input_shape=(None, train_ds.shape[1]))
model.summary()


In [ ]:
# -------- 7) Entraînement avec early stopping -------------------
early = callbacks.EarlyStopping(
    monitor="val_mae", patience=15, restore_best_weights=True
)

hist = model.fit(
    train_ds, train_labels,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    verbose=0,                         # pas de log détaillé
    callbacks=[early, tfdocs.modeling.EpochDots()]
)


In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_ds, test_labels, verbose=0)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_ds).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
